# Lab 2 — Task 1: Music Genre Classification with CNNs

Notebook dedicado exclusivamente a la **Task 1**: entrenar una CNN desde cero y comparar configuraciones de arquitectura e hiperparámetros. La **Task 2 (VGG16)** está en el notebook independiente `Lab2_Task2_VGG16_Music_Genre.ipynb`.

**Criterio de evaluación:** se reserva un 20 % para test antes de optimizar. El 80 % restante se divide en entrenamiento y validación. Las configuraciones se comparan usando únicamente validation accuracy; el test se evalúa una sola vez con la configuración elegida.

Los experimentos cambian un factor cada vez respecto a BASE0, excepto los experimentos de profundidad, que cambian la secuencia de filtros. La configuración de mayor resolución se ejecuta al final porque es la más costosa.

## 1. Imports y configuración

TensorFlow/Keras implementa la CNN. Si faltan librerías, descomenta y ejecuta la instalación. Las semillas ayudan a que la partición y los experimentos sean más reproducibles.

In [ ]:
# %pip install tensorflow scikit-learn matplotlib
import os
import random
import gc
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)

## 2. Cargar las imágenes y separar los datos

Cada subcarpeta de images_original representa un género musical. La división inicial reserva el 20 % para test. Del 80 % restante se extrae un conjunto de validación; por tanto, aproximadamente el 64 % de todos los datos se usa para entrenar, el 16 % para validar y el 20 % para test.

In [ ]:
# Ejemplo en Colab: '/content/drive/MyDrive/GTZAN/images_original'
DATA_DIR = 'Data/images_original'  # Ajusta la ruta si los datos están en otro lugar.
BATCH_SIZE = 32
MAX_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 4

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"No encuentro {DATA_DIR}. Cambia DATA_DIR para apuntar a images_original."
    )

class_names = sorted(
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
)
extensions = ('.png', '.jpg', '.jpeg', '.bmp')
paths, labels = [], []

for label, genre in enumerate(class_names):
    folder = os.path.join(DATA_DIR, genre)
    for filename in sorted(os.listdir(folder)):
        if filename.lower().endswith(extensions):
            paths.append(os.path.join(folder, filename))
            labels.append(label)

paths = np.asarray(paths)
labels = np.asarray(labels)

if len(paths) == 0:
    raise ValueError("No se han encontrado imágenes. Comprueba la estructura de carpetas.")

print("Géneros:", class_names)
print("Total de imágenes:", len(paths))

# Test queda separado antes de comenzar cualquier experimento.
dev_paths, test_paths, dev_labels, test_labels = train_test_split(
    paths, labels, test_size=0.20, random_state=SEED, stratify=labels
)
# La validación procede solo del 80 % de desarrollo.
train_paths, val_paths, train_labels, val_labels = train_test_split(
    dev_paths, dev_labels, test_size=0.20, random_state=SEED, stratify=dev_labels
)

print(f"Train: {len(train_paths)} | Validation: {len(val_paths)} | Test: {len(test_paths)}")
print("No se usa test para comparar configuraciones.")

## 3. Preparar datasets para distintas resoluciones

Cada configuración puede usar un tamaño de imagen distinto. Por eso las imágenes se redimensionan al construir los datasets de cada experimento. Solo se baraja el entrenamiento; validación y test mantienen el orden de las etiquetas.

In [ ]:
def load_image(path, label, size):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, size)
    return image, label


def make_dataset(file_paths, file_labels, size, training=False):
    ds = tf.data.Dataset.from_tensor_slices((file_paths, file_labels))
    if training:
        ds = ds.shuffle(
            buffer_size=len(file_paths), seed=SEED, reshuffle_each_iteration=True
        )
    ds = ds.map(
        lambda path, label: load_image(path, label, size),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


# Mostrar algunos ejemplos para comprobar que se cargan las imágenes correctamente.
preview_ds = make_dataset(train_paths, train_labels, size=(128, 192), training=True)
for images, labels_batch in preview_ds.take(1):
    plt.figure(figsize=(9, 3))
    for i in range(min(4, len(images))):
        plt.subplot(1, 4, i + 1)
        plt.imshow(tf.cast(images[i], tf.uint8))
        plt.title(class_names[int(labels_batch[i])])
        plt.axis("off")
    plt.tight_layout()
    plt.show()
del preview_ds

## 4. Definir la CNN y los experimentos

La CNN aplica una secuencia configurable de bloques convolucionales. Cada bloque contiene Conv2D con padding same y MaxPooling2D. Después, GlobalAveragePooling2D y una capa densa forman la cabeza de clasificación. GlobalAveragePooling2D evita que el número de parámetros crezca demasiado al aumentar la resolución.

- BASE0 es la referencia: filtros (32, 64, 128), kernel 3×3, dropout 0.3, augmentation activado, learning rate 0.001 y resolución 128×192.
- DEPTH_EXPERIMENTS cambia la profundidad o el número de filtros.
- FACTOR_EXPERIMENTS cambia un único factor cada vez respecto a BASE0.
- La opción 3conv_32-64-128 de DEPTH_EXPERIMENTS es idéntica a BASE0 en los parámetros definidos; se omite como duplicado para no gastar tiempo en un experimento redundante.
- Los kernels se expresan como (alto, ancho). En espectrogramas, el primer valor puede interpretarse como extensión vertical en frecuencia y el segundo como extensión horizontal en tiempo.

In [ ]:
BASE0 = dict(
    filters=(32, 64, 128),
    kernel=(3, 3),
    dropout=0.3,
    augment=True,
    lr=1e-3,
    size=(128, 192)
)

DEPTH_EXPERIMENTS = {
    '2conv_16-32':             dict(filters=(16, 32)),
    '3conv_16-32-64':          dict(filters=(16, 32, 64)),
    '3conv_32-64-128':         dict(filters=(32, 64, 128)),  # Igual que BASE0.
    '4conv_16-32-64-128':      dict(filters=(16, 32, 64, 128)),
    '4conv_32-64-128-256':     dict(filters=(32, 64, 128, 256)),
    '5conv_32-64-128-256-256': dict(filters=(32, 64, 128, 256, 256)),
}

FACTOR_EXPERIMENTS = {
    'dropout_0.0':     dict(dropout=0.0),
    'dropout_0.5':     dict(dropout=0.5),
    'no_augment':      dict(augment=False),
    'lr_3e-4':         dict(lr=3e-4),
    'lr_3e-3':         dict(lr=3e-3),
    'kernel_5x5':      dict(kernel=(5, 5)),
    'kernel_7x3_freq': dict(kernel=(7, 3)),
    'kernel_3x7_time': dict(kernel=(3, 7)),
    'size_64x96':      dict(size=(64, 96)),
    'size_192x288':    dict(size=(192, 288)),  # Más costoso: se ejecuta el último.
}

def merge_config(base, changes):
    config = base.copy()
    config.update(changes)
    return config

configurations = {'BASE0': BASE0.copy()}

# Añadir experimentos de profundidad, omitiendo el duplicado de BASE0.
for name, changes in DEPTH_EXPERIMENTS.items():
    config = merge_config(BASE0, changes)
    if config != BASE0:
        configurations[name] = config

# Añadir experimentos que modifican un único factor de BASE0.
for name, changes in FACTOR_EXPERIMENTS.items():
    configurations[name] = merge_config(BASE0, changes)

print(f"Número de configuraciones únicas: {len(configurations)}")
for name, config in configurations.items():
    print(f"{name}: {config}")

## 5. Entrenar y seleccionar la configuración

La función generate_model recibe todos los parámetros de los experimentos. Augmentation solo se aplica durante el entrenamiento. Para cada configuración, EarlyStopping vigila validation accuracy y recupera los mejores pesos.

Para evitar consumir demasiada memoria, no se mantienen todos los modelos en memoria: se guardan las curvas de entrenamiento y los pesos de la mejor configuración según validación. El test no se consulta durante este proceso.

In [ ]:
def generate_model(num_classes, config):
    height, width = config['size']
    inputs = tf.keras.Input(shape=(height, width, 3))
    x = inputs

    # Augmentation se activa solo si lo indica la configuración.
    if config['augment']:
        x = tf.keras.layers.RandomTranslation(0.05, 0.05)(x)
        x = tf.keras.layers.RandomZoom(0.1)(x)
        x = tf.keras.layers.RandomContrast(0.1)(x)

    # Normalizar los píxeles a [0, 1].
    x = tf.keras.layers.Rescaling(1.0 / 255.0)(x)

    for n_filters in config['filters']:
        x = tf.keras.layers.Conv2D(
            filters=n_filters,
            kernel_size=config['kernel'],
            padding='same',
            activation='relu'
        )(x)
        x = tf.keras.layers.MaxPooling2D(pool_size=2)(x)

    # Reduce los mapas de características sin crear una capa Flatten enorme.
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    if config['dropout'] > 0:
        x = tf.keras.layers.Dropout(config['dropout'])(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=config['lr']),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


num_classes = len(class_names)
experiment_results = {}
best_name = None
best_val_accuracy = -1.0
best_weights = None
best_config = None

for index, (name, config) in enumerate(configurations.items(), start=1):
    print(f"\n[{index}/{len(configurations)}] Entrenando {name}: {config}")
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(SEED)

    # La resolución se adapta a cada experimento; los índices de las imágenes no cambian.
    train_ds = make_dataset(train_paths, train_labels, config['size'], training=True)
    val_ds = make_dataset(val_paths, val_labels, config['size'])

    candidate = generate_model(num_classes, config)
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        mode='max',
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True
    )
    history = candidate.fit(
        train_ds,
        validation_data=val_ds,
        epochs=MAX_EPOCHS,
        shuffle=False,  # train_ds ya se baraja con Dataset.shuffle().
        callbacks=[early_stop],
        verbose=1
    )

    run_val_accuracy = float(max(history.history['val_accuracy']))
    experiment_results[name] = {
        'config': config.copy(),
        'history': {key: list(values) for key, values in history.history.items()},
        'val_accuracy': run_val_accuracy,
        'epochs': len(history.history['loss'])
    }
    print(f"{name}: mejor validation accuracy = {run_val_accuracy:.4f}")

    # La selección usa solo validación; el test permanece intacto.
    if run_val_accuracy > best_val_accuracy:
        best_val_accuracy = run_val_accuracy
        best_name = name
        best_config = config.copy()
        best_weights = [weight.copy() for weight in candidate.get_weights()]

    del candidate, history, train_ds, val_ds
    tf.keras.backend.clear_session()
    gc.collect()

print("\nConfiguración seleccionada por validation accuracy:", best_name)
print("Mejor validation accuracy:", f"{best_val_accuracy:.4f}")
print("Parámetros seleccionados:", best_config)

## 6. Comparar los experimentos y evaluar el modelo seleccionado

La tabla incluye todas las configuraciones y su validation accuracy. El conjunto de test se usa una sola vez, después de seleccionar la configuración. Así se evita escoger un modelo indirectamente por su resultado en test.

In [ ]:
# Tabla ordenada por validation accuracy. No incluye test accuracy por configuración,
# porque el test no debe usarse para comparar ni seleccionar modelos.
sorted_results = sorted(
    experiment_results.items(),
    key=lambda item: item[1]['val_accuracy'],
    reverse=True
)

print(f"{'Configuración':34s} {'Val accuracy':>12s} {'Épocas':>8s}  Parámetros")
print("-" * 120)
for name, run in sorted_results:
    print(
        f"{name:34s} {run['val_accuracy']:12.4f} {run['epochs']:8d}  "
        f"{run['config']}"
    )

# Reconstruir únicamente el mejor modelo y restaurar los pesos guardados.
tf.keras.backend.clear_session()
model = generate_model(num_classes, best_config)
model.set_weights(best_weights)

test_ds = make_dataset(test_paths, test_labels, best_config['size'])
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
print(f"\nModelo seleccionado: {best_name}")
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

# Métricas por género y matriz de confusión sobre el test reservado.
y_true = test_labels
y_prob = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print("\nInforme de clasificación en test:")
print(classification_report(
    y_true, y_pred,
    labels=range(num_classes),
    target_names=class_names,
    zero_division=0
))

cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax, xticks_rotation=45, cmap='Blues', colorbar=False
)
ax.set_title(f'Matriz de confusión en test — {best_name}')
plt.tight_layout()
plt.show()

# Gráfica de validation accuracy para todas las configuraciones.
names = [name for name, _ in sorted_results]
scores = [run['val_accuracy'] for _, run in sorted_results]
plt.figure(figsize=(12, max(5, len(names) * 0.35)))
plt.barh(names[::-1], scores[::-1])
plt.xlabel('Best validation accuracy')
plt.title('Comparación de configuraciones (validación)')
plt.xlim(0, 1)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# Curvas de entrenamiento de la configuración seleccionada.
selected_history = experiment_results[best_name]['history']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(selected_history['accuracy'], marker='o', label='Train')
axes[0].plot(selected_history['val_accuracy'], marker='x', label='Validation')
axes[0].set(title=f'{best_name} — Accuracy', xlabel='Época', ylabel='Accuracy')
axes[1].plot(selected_history['loss'], marker='o', label='Train')
axes[1].plot(selected_history['val_loss'], marker='x', label='Validation')
axes[1].set(title=f'{best_name} — Loss', xlabel='Época', ylabel='Loss')
for ax in axes:
    ax.legend()
    ax.grid(True)
plt.tight_layout()
plt.show()

## 7. Discusión y conclusión

Después de ejecutar el notebook, comenta:
- Qué configuración obtiene la mayor validation accuracy y si la diferencia respecto a BASE0 es relevante.
- Cómo afectan la profundidad, el dropout, augmentation, learning rate, kernel y resolución.
- Si las curvas de entrenamiento y validación sugieren overfitting o underfitting.
- La test accuracy y los géneros que más se confunden, según la matriz de confusión.
- Las diferencias con Lab 1, comprobando que las particiones y los datos sean comparables.

Los experimentos de profundidad cambian la secuencia de filtros; los experimentos FACTOR cambian una sola propiedad respecto a BASE0. Aun así, los resultados dependen de la inicialización y del conjunto de validación. Para conclusiones sólidas convendría repetir experimentos con varias semillas.

**Nota metodológica:** la partición actual es estratificada por imagen. Si existen varios espectrogramas derivados de una misma canción, lo más riguroso es separar por canción original para evitar que imágenes relacionadas aparezcan en entrenamiento y test.

## Notas de ejecución

1. Ajusta DATA_DIR a la carpeta images_original.
2. Ejecuta las celdas en orden.
3. Se ejecutan 16 configuraciones únicas; la variante 192×288 se deja al final por su coste computacional.
4. La selección se realiza con validation accuracy. El test reservado se evalúa solo para el modelo seleccionado.
5. Task 2 está completamente separada en el notebook Lab2_Task2_VGG16_Music_Genre.ipynb.

Los mensajes de CUDA/cuDNN o oneDNN suelen ser informativos sobre el hardware disponible; no implican por sí mismos un error en el modelo.